# Study 1 full scoring run: CheXzero ensemble and CLEAR on NIH ChestX-ray14

Follows the OSF preregistration (posted 1 October 2026, embargoed). This notebook only **produces scores**; it computes no AUROC or subgroup result. The analysis is done afterwards from the files it saves.

What it does:
1. Builds the cohort: official test split (ages over 100 excluded) plus held-out training patients (10%, seed 2026) for thresholds.
2. Applies the eligibility rule from labels only (at least 30 positive images from at least 10 patients in every subgroup).
3. Embeds the 28 label prompts with SFR-Embedding-Mistral as plain text, as in CLEAR's published code, and also with the instruction prefix that reproduces the authors' released concept embeddings.
4. Scores every image with the 10-checkpoint CheXzero ensemble, CLEAR concept-based (primary) and CLEAR direct (secondary).
5. Packs everything into `study1_outputs.zip`.

**Settings:** Accelerator **GPU T4 x2** (needed for the 7-billion-parameter text model), Internet **On**, datasets **NIH Chest X-rays** and your private **CheXzero weights** dataset attached.
**How to run:** use **Save Version → Save & Run All (Commit)** so it runs in the background (about 2 to 3 hours) and the outputs are kept. Then open the saved version, go to its **Output** tab and download `study1_outputs.zip`.

In [ ]:
# 0. Settings
import os, sys, time, json, glob, subprocess, hashlib, platform
os.environ["HF_HOME"] = "/tmp/hf"          # large downloads go to /tmp, not the 20 GB output folder
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
SEED = 2026
OUT = "/kaggle/working/study1_out"; os.makedirs(OUT, exist_ok=True)
FINDINGS = ["Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule","Pneumonia",
            "Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis","Pleural_Thickening","Hernia"]
PROMPT_WORDS = {"Atelectasis":"atelectasis","Cardiomegaly":"cardiomegaly","Effusion":"pleural effusion",
                "Infiltration":"infiltration","Mass":"mass","Nodule":"nodule","Pneumonia":"pneumonia",
                "Pneumothorax":"pneumothorax","Consolidation":"consolidation","Edema":"edema",
                "Emphysema":"emphysema","Fibrosis":"fibrosis","Pleural_Thickening":"pleural thickening",
                "Hernia":"hernia"}
POS = [PROMPT_WORDS[f] for f in FINDINGS]
NEG = ["no " + w for w in POS]
run_info = {"started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), "seed": SEED,
            "prompts_pos": POS, "prompts_neg": NEG}
T0 = time.time()

In [ ]:
# 1. Packages and versions
!pip install -q git+https://github.com/openai/CLIP.git gdown
!rm -rf /kaggle/working/CLEAR && git clone -q https://github.com/peterhan91/CLEAR.git /kaggle/working/CLEAR
!pip install -q /kaggle/working/CLEAR
import torch, numpy as np, pandas as pd, transformers
run_info["clear_git_commit"] = subprocess.run(["git","-C","/kaggle/working/CLEAR","rev-parse","HEAD"],
                                              capture_output=True, text=True).stdout.strip()
try:
    from huggingface_hub import HfApi
    run_info["clear_hf_sha"] = HfApi().model_info("peterhan91/CLEAR").sha
except Exception as e:
    run_info["clear_hf_sha"] = "unavailable: " + repr(e)[:200]
run_info.update({"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__,
                 "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
print(json.dumps(run_info, indent=1))
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2"

In [ ]:
# 2. Cohort: official test split + 10% held-out training patients
def find_file(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if not d.startswith("images_")]
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name)
entry_csv = find_file("Data_Entry_2017.csv"); base = os.path.dirname(entry_csv)
img_paths = {os.path.basename(p): p for p in glob.glob(os.path.join(base, "images_*", "images", "*.png"))}
print("images:", len(img_paths))

df = pd.read_csv(entry_csv).rename(columns={"Image Index":"image","Finding Labels":"labels",
        "Patient ID":"patient","Patient Age":"age","Patient Gender":"sex","View Position":"view"})
df = df[["image","labels","patient","age","sex","view"]]
test_ids = set(open(os.path.join(base, "test_list.txt")).read().split())
tv_ids = set(open(os.path.join(base, "train_val_list.txt")).read().split())
for f in FINDINGS:
    df[f] = df.labels.str.contains(f, regex=False).astype(int)
df = df[df.age <= 100].copy()
df["age_band4"] = pd.cut(df.age, [-1, 18, 40, 60, 200], labels=["0-18","18-40","40-60","60+"]).astype(str)
df["age_band5"] = pd.cut(df.age, [-1, 18, 40, 60, 80, 200], labels=["0-18","18-40","40-60","60-80","80+"]).astype(str)

rng = np.random.default_rng(SEED)
tv_patients = np.array(sorted(df[df.image.isin(tv_ids)].patient.unique()))
held_patients = set(rng.choice(tv_patients, size=int(round(0.10 * len(tv_patients))), replace=False).tolist())
df["set"] = np.where(df.image.isin(test_ids), "test",
             np.where(df.image.isin(tv_ids) & df.patient.isin(held_patients), "heldout", "unused"))
cohort = df[df.set != "unused"].reset_index(drop=True)
cohort["path"] = cohort.image.map(img_paths)
assert cohort.path.notna().all(), "some images not found"
assert not (set(cohort[cohort.set=="test"].patient) & set(cohort[cohort.set=="heldout"].patient))
cohort.drop(columns=["path"]).to_csv(f"{OUT}/cohort.csv", index=False)
run_info["n_test_images"] = int((cohort.set=="test").sum()); run_info["n_test_patients"] = int(cohort[cohort.set=="test"].patient.nunique())
run_info["n_heldout_images"] = int((cohort.set=="heldout").sum()); run_info["n_heldout_patients"] = len(held_patients)
print({k: run_info[k] for k in ["n_test_images","n_test_patients","n_heldout_images","n_heldout_patients"]})

In [ ]:
# 3. Eligibility from labels only (no model output involved)
test = cohort[cohort.set=="test"]
def eligible(data, group_col, groups):
    out, detail = [], {}
    for f in FINDINGS:
        ok = True; detail[f] = {}
        for g in groups:
            sub = data[(data[group_col]==g) & (data[f]==1)]
            n_img, n_pat = len(sub), sub.patient.nunique()
            detail[f][g] = [int(n_img), int(n_pat)]
            ok &= (n_img >= 30) and (n_pat >= 10)
        if ok: out.append(f)
    return out, detail
pa = test[test.view=="PA"]
elig = {}
for name, data, col, groups in [("sex", test, "sex", ["F","M"]), ("age", test, "age_band4", ["0-18","18-40","40-60","60+"]),
                                ("sex_PA", pa, "sex", ["F","M"]), ("age_PA", pa, "age_band4", ["0-18","18-40","40-60","60+"])]:
    lst, det = eligible(data, col, groups)
    elig[name] = {"eligible": lst, "positives_images_patients": det}
    print(f"{name}: {len(lst)} eligible ->", lst)
json.dump(elig, open(f"{OUT}/eligibility.json","w"), indent=1)

In [ ]:
# 4. Prompt embeddings with SFR-Embedding-Mistral, following CLEAR's get_embed.py
#    (AutoModel, fp16, last-token pooling, L2 normalisation). This runs in a separate process so
#    that all its GPU memory is released when it finishes. Each text is embedded on its own (no padding).
#    Two prompt versions are saved: plain text, as in the authors' exp_zeroshot.py (primary), and with
#    the instruction prefix that reproduces the authors' released concept embeddings (secondary).
import gc
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from clear.concepts import ConceptBank
concepts_csv = hf_hub_download("peterhan91/CLEAR", "mimic_concepts.csv")
clip_pt = hf_hub_download("peterhan91/CLEAR", "concept_embeddings_368294.pt")
llm_pkl = hf_hub_download("peterhan91/CLEAR", "concepts_embeddings_sfr_mistral.pickle")
bank = ConceptBank.from_files(concepts_csv, clip_embeddings_path=clip_pt, llm_embeddings_path=llm_pkl,
                              allow_unsafe_pickle=True)
C_llm = bank.ensure_llm_embeddings()
pick = np.random.default_rng(SEED).choice(bank.num_concepts, 40, replace=False)
INSTR = "Instruct: Given medical radiology concepts, retrieve relevant passages and generate embeddings\nQuery: "
json.dump({"pos": POS, "neg": NEG, "check": [bank.concepts[i] for i in pick], "instr": INSTR},
          open("/kaggle/working/sfr_in.json", "w"))
SFR_SCRIPT = r'''
import json, torch, numpy as np, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
d = json.load(open("/kaggle/working/sfr_in.json"))
tok = AutoTokenizer.from_pretrained("Salesforce/SFR-Embedding-Mistral")
mm = {i: "13GiB" for i in range(torch.cuda.device_count())}; mm["cpu"] = "24GiB"
llm = AutoModel.from_pretrained("Salesforce/SFR-Embedding-Mistral", dtype=torch.float16,
                                device_map="auto", max_memory=mm).eval()
dev0 = next(llm.parameters()).device
def pool(h, m):
    if m[:, -1].sum() == m.shape[0]:
        return h[:, -1]
    return h[torch.arange(h.shape[0], device=h.device), m.sum(1) - 1]
@torch.no_grad()
def emb(texts):
    out = []
    for s in texts:
        e = tok(s, max_length=8192, truncation=True, return_tensors="pt").to(dev0)
        out.append(F.normalize(pool(llm(**e).last_hidden_state, e["attention_mask"]).float(), dim=-1).cpu())
    return torch.cat(out).numpy()
I = d["instr"]
np.savez("/kaggle/working/sfr_out.npz",
         pos=emb(d["pos"]), neg=emb(d["neg"]),
         pos_i=emb([I + s for s in d["pos"]]), neg_i=emb([I + s for s in d["neg"]]),
         chk=emb(d["check"]), chk_i=emb([I + s for s in d["check"]]))
print("SFR embeddings done")
'''
open("/kaggle/working/sfr_embed.py", "w").write(SFR_SCRIPT)
t = time.time()
r = subprocess.run([sys.executable, "/kaggle/working/sfr_embed.py"], capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-3000:])
assert r.returncode == 0, "SFR embedding step failed"
z = np.load("/kaggle/working/sfr_out.npz")
pos_llm, neg_llm, pos_llm_i, neg_llm_i = z["pos"], z["neg"], z["pos_i"], z["neg_i"]
for k in ["pos", "neg", "pos_i", "neg_i"]:
    np.save(f"{OUT}/prompt_emb_sfr_{k}.npy", z[k])
ref = C_llm[pick] / np.linalg.norm(C_llm[pick], axis=1, keepdims=True)
cos_plain = (z["chk"] * ref).sum(1); cos_instr = (z["chk_i"] * ref).sum(1)
run_info["sfr_check_cos_plain_mean_min"] = [float(cos_plain.mean()), float(cos_plain.min())]
run_info["sfr_check_cos_instruct_mean_min"] = [float(cos_instr.mean()), float(cos_instr.min())]
run_info["n_concepts"] = bank.num_concepts; run_info["llm_dim"] = int(C_llm.shape[1])
print("Reproduction of released concept embeddings, cosine mean/min: plain", run_info["sfr_check_cos_plain_mean_min"],
      "| with instruction", run_info["sfr_check_cos_instruct_mean_min"])
subprocess.run(["rm", "-rf", "/tmp/hf/hub/models--Salesforce--SFR-Embedding-Mistral"])
gc.collect(); torch.cuda.empty_cache()
print("GPU memory in use after step 4 (GB):", [round(torch.cuda.memory_allocated(i)/1e9, 2) for i in range(torch.cuda.device_count())])
run_info["sec_prompt_embedding"] = round(time.time()-t)

In [ ]:
# 5. Load CLEAR and build the concept projection.
#    The authors' pipeline computes  normalise( (img @ norm(C_clip).T) @ C_llm ).
#    Because this is linear, it equals  normalise( img @ M ) with M = norm(C_clip).T @ C_llm,
#    which is computed once. Both forms are compared on real images in step 7.
from clear import load_pretrained, tokenize
dev = "cuda:0"
dev_cz = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
clear_model, clear_pre = load_pretrained(device=dev); clear_model.eval()
C_clip = torch.from_numpy(bank.ensure_clip_embeddings()).float()

# check that the released concept CLIP embeddings match CLEAR's text encoder (500 random concepts)
pick2 = np.random.default_rng(SEED+1).choice(bank.num_concepts, 500, replace=False)
with torch.no_grad():
    enc = F.normalize(clear_model.encode_text(tokenize([bank.concepts[i] for i in pick2]).to(dev)).float(), dim=-1).cpu()
cos_clip = (enc * F.normalize(C_clip[pick2], dim=-1)).sum(1)
run_info["concept_clip_check_cos_mean_min"] = [float(cos_clip.mean()), float(cos_clip.min())]
print("Released concept CLIP embeddings vs text encoder, cosine mean/min:", run_info["concept_clip_check_cos_mean_min"])
if cos_clip.min() < 0.999:
    print("Mismatch: recomputing all concept CLIP embeddings with the text encoder")
    chunks = []
    with torch.no_grad():
        for i in range(0, bank.num_concepts, 2048):
            chunks.append(clear_model.encode_text(tokenize(list(bank.concepts[i:i+2048])).to(dev)).float().cpu())
    C_clip = torch.cat(chunks); run_info["concept_clip_recomputed"] = True
else:
    run_info["concept_clip_recomputed"] = False

M = torch.zeros(C_clip.shape[1], C_llm.shape[1], dtype=torch.float64, device=dev)
C_llm_t = torch.from_numpy(C_llm)
for i in range(0, C_clip.shape[0], 8192):
    a = F.normalize(C_clip[i:i+8192].to(dev), dim=1).double()
    M += a.T @ C_llm_t[i:i+8192].to(dev).double()
M = M.float()
np.save(f"{OUT}/clear_concept_projection_M.npy", M.cpu().numpy())
pos_t = F.normalize(torch.from_numpy(pos_llm).to(dev), dim=-1); neg_t = F.normalize(torch.from_numpy(neg_llm).to(dev), dim=-1)
pos_ti = F.normalize(torch.from_numpy(pos_llm_i).to(dev), dim=-1); neg_ti = F.normalize(torch.from_numpy(neg_llm_i).to(dev), dim=-1)
with torch.no_grad():
    cpos = F.normalize(clear_model.encode_text(tokenize(POS).to(dev)).float(), dim=-1)
    cneg = F.normalize(clear_model.encode_text(tokenize(NEG).to(dev)).float(), dim=-1)
print("M:", tuple(M.shape))

In [ ]:
# 6. Load the 10 CheXzero checkpoints: from an attached Kaggle dataset if present, otherwise Google Drive
import gdown, clip
from torchvision.transforms import Compose, Normalize, Resize, InterpolationMode
def find_pts():
    hits = []
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if not d.startswith("images_")]
        hits += [os.path.join(root, f) for f in files if f.endswith(".pt")]
    return sorted(hits)
ckpts = find_pts()
if len(ckpts) < 10:
    os.makedirs("/kaggle/working/chexzero_weights", exist_ok=True)
    gdown.download_folder("https://drive.google.com/drive/folders/1makFLiEMbSleYltaRxw81aBhEDMpVwno",
                          output="/kaggle/working/chexzero_weights", quiet=True, remaining_ok=True)
    ckpts = sorted(glob.glob("/kaggle/working/chexzero_weights/**/*.pt", recursive=True))
assert len(ckpts) == 10, f"expected 10 CheXzero checkpoints, found {len(ckpts)}: {ckpts}"
print("CheXzero checkpoints from:", os.path.dirname(ckpts[0]))
run_info["chexzero_checkpoints"] = [[os.path.basename(c), os.path.getsize(c)] for c in ckpts]
cz_models, cz_text = [], []
for c in ckpts:
    m, _ = clip.load("ViT-B/32", device="cpu", jit=False)
    m.load_state_dict(torch.load(c, map_location="cpu")); m = m.to(dev_cz).eval()
    with torch.no_grad():
        tp = F.normalize(m.encode_text(clip.tokenize(POS, context_length=77).to(dev_cz)).float(), dim=-1)
        tn = F.normalize(m.encode_text(clip.tokenize(NEG, context_length=77).to(dev_cz)).float(), dim=-1)
    cz_models.append(m); cz_text.append((tp, tn))
print("CheXzero checkpoints loaded:", len(cz_models), "| GPU GB in use:", [round(torch.cuda.memory_allocated(i)/1e9, 2) for i in range(torch.cuda.device_count())])

In [ ]:
# 7. One pass over all cohort images: CheXzero ensemble, CLEAR direct, CLEAR concept
from PIL import Image
from torch.utils.data import Dataset, DataLoader
cz_tf = Compose([Normalize((101.48761,)*3, (83.43944,)*3), Resize(224, interpolation=InterpolationMode.BICUBIC)])
class DS(Dataset):
    def __init__(self, paths): self.p = list(paths)
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        im = Image.open(self.p[i])
        g = np.asarray(im.convert("L").resize((320, 320), Image.BICUBIC), dtype=np.float32)
        x_cz = cz_tf(torch.from_numpy(np.repeat(g[None], 3, axis=0)))
        x_cl = clear_pre(im.convert("RGB"))
        return x_cz, x_cl
BS = 32
dl = DataLoader(DS(cohort.path), batch_size=BS, num_workers=4, pin_memory=True)

def pair_prob(f, tp, tn):
    p, n = f @ tp.T, f @ tn.T
    return torch.exp(p) / (torch.exp(p) + torch.exp(n))

from clear.pipeline import ZeroShotPipeline
cz_all, cz_per, cl_dir, cl_con, cl_con_i, cl_feat = [], [], [], [], [], []
checked = False; t = time.time()
with torch.no_grad():
    for b, (x_cz, x_cl) in enumerate(dl):
        x_cz, x_cl = x_cz.to(dev_cz, non_blocking=True), x_cl.to(dev, non_blocking=True)
        per = []
        for m, (tp, tn) in zip(cz_models, cz_text):
            f = F.normalize(m.encode_image(x_cz).float(), dim=-1)
            per.append(pair_prob(f, tp, tn))
        per = torch.stack(per, 0)                          # 10 x B x 14
        cz_per.append(per.cpu().numpy()); cz_all.append(per.mean(0).cpu().numpy())
        fi = F.normalize(clear_model.encode_image(x_cl).float(), dim=-1)
        cl_feat.append(fi.half().cpu().numpy())
        cl_dir.append(pair_prob(fi, cpos, cneg).cpu().numpy())
        rep = F.normalize(fi @ M, dim=-1)
        cl_con.append(pair_prob(rep, pos_t, neg_t).cpu().numpy())
        cl_con_i.append(pair_prob(rep, pos_ti, neg_ti).cpu().numpy())
        if not checked:   # compare the fast projection with the authors' chunked pipeline on this batch
            pipe = ZeroShotPipeline(clear_model, ConceptBank(concepts=bank.concepts, clip_embeddings=C_clip.numpy(),
                                    llm_embeddings=C_llm), device=torch.device(dev))
            ref = pipe.project_to_concept_space(fi)
            run_info["projection_check_max_abs_diff"] = float((ref.float() - rep).abs().max())
            print("Projection check, max abs difference vs authors' pipeline:", run_info["projection_check_max_abs_diff"])
            del pipe; checked = True
        if b % 50 == 0:
            done = (b+1)*BS
            print(f"{min(done, len(cohort))}/{len(cohort)} images, {round((time.time()-t)/60,1)} min")
run_info["min_scoring"] = round((time.time()-t)/60, 1)

In [ ]:
# 8. Save scores (one row per image, same order as cohort.csv)
def save(arr, name):
    d = pd.DataFrame(np.concatenate(arr), columns=FINDINGS)
    d.insert(0, "image", cohort.image.values); d.insert(1, "set", cohort.set.values)
    d.to_csv(f"{OUT}/{name}.csv", index=False, float_format="%.6f"); return d
s_cz = save(cz_all, "scores_chexzero_ensemble")
s_cd = save(cl_dir, "scores_clear_direct")
s_cc = save(cl_con, "scores_clear_concept")
s_ci = save(cl_con_i, "scores_clear_concept_instructprompts")
np.save(f"{OUT}/scores_chexzero_per_checkpoint.npy", np.concatenate(cz_per, axis=1).astype(np.float32))  # 10 x N x 14
np.save("/kaggle/working/clear_image_features_fp16.npy", np.concatenate(cl_feat))  # kept outside the zip

# sanity checks only: ranges, missing values, agreement between score sets (no labels used)
for name, d in [("chexzero", s_cz), ("clear_direct", s_cd), ("clear_concept", s_cc), ("clear_concept_instr", s_ci)]:
    v = d[FINDINGS].values
    run_info[f"{name}_nan"] = int(np.isnan(v).sum()); run_info[f"{name}_range"] = [float(v.min()), float(v.max())]
run_info["corr_chexzero_vs_clear_concept"] = float(np.corrcoef(s_cz[FINDINGS].values.ravel(), s_cc[FINDINGS].values.ravel())[0,1])
run_info["corr_clear_direct_vs_concept"] = float(np.corrcoef(s_cd[FINDINGS].values.ravel(), s_cc[FINDINGS].values.ravel())[0,1])
run_info["corr_clear_concept_plain_vs_instr"] = float(np.corrcoef(s_cc[FINDINGS].values.ravel(), s_ci[FINDINGS].values.ravel())[0,1])
run_info["total_min"] = round((time.time()-T0)/60, 1)
json.dump(run_info, open(f"{OUT}/run_info.json","w"), indent=1)
!cd /kaggle/working && rm -f study1_outputs.zip && zip -qr study1_outputs.zip study1_out
print("Saved /kaggle/working/study1_outputs.zip", round(os.path.getsize("/kaggle/working/study1_outputs.zip")/1e6,1), "MB")
print("===== RUN SUMMARY =====")
print(json.dumps({k: v for k, v in run_info.items() if k not in ("chexzero_checkpoints",)}, indent=1))
print("===== END =====")